In [ ]:
# SPEC §8 E2, seed 1 only (decision 2026-10-28: E5 has the laptop; seed 0 finished there, reported as a shortfall).
# Needs MyDrive/mtp/ur5rl_bundle.zip from 2026-10-28 (carries configs/exp/e2.yaml: SAC, sparse+HER and dense).
# CPU runtime. Runs sac_sparse_her s1 then sac_dense s1, one at a time (~80 min each at ~105 steps/s on Colab).
# Checkpoints go to MyDrive/mtp/runs every 25k steps; finished runs are skipped, so after a disconnect just run this
# cell again. Run ids match the laptop's (sac_sparse_her_a337c4b7_s1, sac_dense_55944f05_s1): when both are done,
# copy MyDrive/mtp/runs/e2/*_s1 into results/runs/e2 on the laptop (replacing the partial local s1 folders).
SEED = 1
import os, subprocess, yaml
from google.colab import drive
drive.mount('/content/drive')
D, W = '/content/drive/MyDrive/mtp', '/content/ur5rl'
sh = lambda c: subprocess.run(c, shell=True, check=True)
if not os.path.exists(f'{W}/scripts/train.py'):  # fresh VM: unpack and install once
    sh(f'mkdir -p {W} && unzip -q -o {D}/ur5rl_bundle.zip -d {W}')
    sh('pip install -q --index-url https://download.pytorch.org/whl/cpu torch==2.14.1')
    sh('pip install -q mujoco==3.17.0 gymnasium==1.4.0 stable-baselines3==2.9.2 numpy==2.4.6 scipy==1.17.1 '
       'PyYAML==6.0.3 psutil==7.2.2 tensorboard==2.21.0')
    sh(f'pip install -q -e {W} --no-deps --ignore-requires-python')
exp = yaml.safe_load(open(f'{W}/configs/exp/e2.yaml'))
for run in exp['runs']:
    print(f"=== E2 {run['name']} seed {SEED}", flush=True)
    sh(f"cd {W} && python scripts/train.py --exp configs/exp/e2.yaml --run {run['name']} --seed {SEED} "
       f'--threads 2 --out {D}/runs --ckpt-every 25000 2>&1 | grep --line-buffered -E "val]|curriculum|resumed|done|Error|Traceback"')
print('finished; copy', f'{D}/runs/e2/*_s{SEED}', 'into results/runs/e2 on the laptop')
